### Library

In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.layers import Flatten,Dense
import keras_tuner as kt
import numpy as np
from model import build_model

In [8]:
(train_ds, val_ds, test_ds), ds_info = tfds.load(
    'mnist', 
    split=['train[:80%]', 'train[80%:]', 'test'], 
    as_supervised=True,
    with_info=True
)

### Dataset pipeline

In [9]:
def normalize_img(ds, y):
    return tf.cast(ds, tf.float32) / 255., y

BATCH_SIZE=128

train_ds = train_ds.map(normalize_img, num_parallel_calls=tf.data.AUTOTUNE).cache().shuffle(len(train_ds)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(normalize_img, num_parallel_calls=tf.data.AUTOTUNE).cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(normalize_img, num_parallel_calls=tf.data.AUTOTUNE).cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


### Train

In [17]:
tuner = kt.Hyperband(hypermodel=build_model, objective='val_accuracy',  max_epochs=10,factor=3,directory='logs',project_name='mnist_tuning')


In [18]:
stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)

In [19]:
tuner.search(train_ds, epochs=5, validation_data=val_ds, callbacks=[stop_early])

Trial 30 Complete [00h 00m 09s]
val_accuracy: 0.9738333225250244

Best val_accuracy So Far: 0.9770833253860474
Total elapsed time: 00h 02m 32s


In [24]:
tuner.results_summary(num_trials=1)

# 2. Extract the absolute best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"Optimal Learning Rate: {best_hps.get('learning_rate')}")
print(f"Optimal Weight Decay: {best_hps.get('weight_decay')}")

Results summary
Results in logs/mnist_tuning
Showing 1 best trials
Objective(name="val_accuracy", direction="max")

Trial 0016 summary
Hyperparameters:
learning_rate: 0.002357173956586594
weight_decay: 0.0005283071526371937
tuner/epochs: 10
tuner/initial_epoch: 4
tuner/bracket: 2
tuner/round: 2
tuner/trial_id: 0013
Score: 0.9770833253860474
Optimal Learning Rate: 0.002357173956586594
Optimal Weight Decay: 0.0005283071526371937


In [25]:
model = tuner.hypermodel.build(best_hps)

/Users/dungtc/Programming/dl-labs/.venv/lib/python3.10/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [29]:
cb = tf.keras.callbacks.ModelCheckpoint(filepath='./cp.weights.h5', save_weights_only=True, verbose=1)

history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=20,
  callbacks=[cb]
)

Epoch 1/20
353/375 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9959 - loss: 0.0137
Epoch 1: saving model to ./cp.weights.h5
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9951 - loss: 0.0161 - val_accuracy: 0.9756 - val_loss: 0.0906
Epoch 2/20
365/375 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9964 - loss: 0.0119
Epoch 2: saving model to ./cp.weights.h5
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9957 - loss: 0.0135 - val_accuracy: 0.9732 - val_loss: 0.1050
Epoch 3/20
372/375 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9967 - loss: 0.0114
Epoch 3: saving model to ./cp.weights.h5
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9966 - loss: 0.0116 - val_accuracy: 0.9738 - val_loss: 0.1084
Epoch 4/20
370/375 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9975 - loss: 0.0091
Epoch 4: saving model to ./cp.weights.h5
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9968 - loss: 0.0102 - val_accuracy: 0.9725 - val_loss: 0.1164
Epoch 5/20
350/375 ━━━━━

In [31]:
test_loss, test_acc = model.evaluate(test_ds, verbose=2)
print('\nTest accuracy:', test_acc)

79/79 - 0s - 1ms/step - accuracy: 0.9787 - loss: 0.1269

Test accuracy: 0.9786999821662903
